# Phase 2 on Kaggle: train, evaluate, explain (Steps 4–6)

**Before running:**
1. **Settings → Accelerator: GPU T4 x2** (or P100). **Internet: ON.**
2. **Add Input**:
   - Datasets → **NIH Chest X-rays** (`nih-chest-xrays/data`), needed for the bounding boxes and as a fallback for missing cache files.
   - Notebooks → your **Phase 1 notebook** (its saved output holds `splits.csv` and the 320 px cache).
3. *Optional, for DagsHub MLflow:* **Add-ons → Secrets**: add `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` and attach them to this notebook. Without them, MLflow logs to `/kaggle/working/mlflow.db`, which is saved with the output.
4. **Save Version → Save & Run All (Commit)**. Expect about 2–3 h; you can close the browser.


In [ ]:
REPO_URL = "https://github.com/<your-username>/MedVision.git"   # <- your repo
REPO_DIR = "MedVision"
!git clone -q $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -q monai pydicom tabulate mlflow

In [ ]:
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
        os.environ[k] = s.get_secret(k)
    print("MLflow -> remote:", os.environ["MLFLOW_TRACKING_URI"])
except Exception:
    for k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
        os.environ.pop(k, None)
    print("MLflow -> local /kaggle/working/mlflow.db")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Link the Phase 1 output
Kaggle stores the Phase 1 output (112k files) as one `_output_.zip`. This extracts it to `/tmp` (local scratch, not re-saved as output), then rewrites the image paths. Takes a few minutes.

In [ ]:
!python scripts/link_phase1.py

## Step 4: smoke test (about 2 min), then full training
The smoke test makes a broken setup fail in minutes instead of hours.

In [ ]:
!python scripts/train.py --epochs 1 --max-steps 20 --subset 640 --run-name smoke

In [ ]:
!python scripts/train.py --run-name densenet121-224

## Step 5: evaluation and calibration

In [ ]:
!python scripts/evaluate.py
from IPython.display import Image, Markdown, display
display(Markdown(open('/kaggle/working/eval/eval_report.md').read()))
for f in ['auroc_ci', 'roc_focus', 'reliability_focus']:
    display(Image(f'/kaggle/working/eval/{f}.png'))

## Step 6: Grad-CAM localisation on NIH bounding boxes

In [ ]:
!python scripts/gradcam_eval.py --split test
display(Markdown(open('/kaggle/working/explain/explain_report.md').read()))
display(Image('/kaggle/working/explain/gradcam_gallery.png'))

## Training curve

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv('/kaggle/working/checkpoints/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
h.plot(x='epoch', y='train_loss', ax=ax[0], marker='o'); h.plot(x='epoch', y='val_mean_auroc', ax=ax[1], marker='o')
plt.tight_layout(); plt.show()
!ls -la /kaggle/working/checkpoints /kaggle/working/eval

Outputs kept in this version: `checkpoints/best.pt`, `eval/calibration.json` (used by the API and report generator), the evaluation and Grad-CAM reports, and the MLflow history.